## Process the glyco data from PXD052447 using IDs and GlyCounter output from glyco_ids.txt file

In [12]:
import pandas as pd 

df = pd.read_csv('data/glyco_ids.txt', sep='\t')
df.fillna(False, inplace=True)

scans = df['Spectrum_ScanNumber'].values
triggered = [x for x in df['Triggered'].values]
gly_class = df['Glycan_Class'].values
labels = [x == 'O-linked' for x in gly_class]



/tmp/ipykernel_184407/292460146.py:3: DtypeWarning: Columns (109,112) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/glyco_ids.txt', sep='\t')
/tmp/ipykernel_184407/292460146.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna(False, inplace=True)
/tmp/ipykernel_184407/292460146.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)


In [16]:
file2scan2info = {}

for scan, scan_trig, label in zip(scans, triggered, labels):
    file = '_'.join(scan.split('_')[:-1])
    scan = scan.split('_')[-1]
    if file not in file2scan2info:
        file2scan2info[file] = {}
    file2scan2info[file][scan] = (label, scan_trig)


In [10]:
from pyteomics import mzml
from tqdm import tqdm 
import numpy as np
with open(f'data/train_pos.mgf','w') as tp_out:
    with open(f'data/train_neg.mgf', 'w') as tn_out:
        with open(f'data/val_pos.mgf','w') as vp_out:
            with open(f'data/val_neg.mgf', 'w') as vn_out:
                with open(f'data/test_pos.mgf','w') as ep_out:
                    with open(f'data/test_neg.mgf', 'w') as en_out:
                        for file in file2scan2info:
                            print(file)
                            found = 0
                            pos = 0
                            if int(file.split('_F')[-1]) % 6 == 0:
                                p_out = ep_out
                                n_out = en_out
                            elif int(file.split('_F')[-1]) % 6 == 1:
                                p_out = vp_out
                                n_out = vn_out
                            else:
                                p_out = tp_out
                                n_out = tn_out
                            with mzml.MzML('data/mzML/'+ file + '.mzML') as reader:
                                for spec in reader:
                                    if spec.get('ms level') == 2:
                                        id = spec['id'].split('scan=')[1]
                                        if id in file2scan2info[file]:
                                            label, scan_trig = file2scan2info[file][id]
                                            found += 1
                                    
                                            window = spec['precursorList']['precursor'][0]
                                            window_center = window['selectedIonList']['selectedIon'][0]['selected ion m/z']
                                            if 'charge state' not in window['selectedIonList']['selectedIon'][0]:
                                                continue
                                            charge = window['selectedIonList']['selectedIon'][0]['charge state']
                                            cur_rt = 60 * spec['scanList']['scan'][0]['scan start time']

                                            mzs = spec['m/z array']
                                            intensities = spec['intensity array']

                                            sorted_intensity_idxs = np.argsort(intensities)[-300:]
                                            intensities = intensities[sorted_intensity_idxs]
                                            mzs = mzs[sorted_intensity_idxs]

                                            sorted_mz_idxs = np.argsort(mzs)
                                            intensities = intensities[sorted_mz_idxs]
                                            mzs = mzs[sorted_mz_idxs]
                                            
                                            if label:
                                                s_out = p_out
                                                pos += 1
                                            else:
                                                s_out = n_out

                                            s_out.write("BEGIN IONS\n")
                                            s_out.write(f"TITLE={str(id)}\n")
                                            s_out.write(f"PEPMASS={window_center}\n")
                                            s_out.write(f"CHARGE={charge}\n")
                                            s_out.write(f"SCANS=F1:{id}\n")
                                            s_out.write(f"RTINSECONDS={cur_rt}\n")
                                            for mz, intensity in zip(mzs, intensities):
                                                s_out.write(f"{mz} {intensity}\n")
                                            s_out.write("END IONS\n")
                                            s_out.write("\n")
                                            
                            print(len(file2scan2info[file]) == found, found, pos)

Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F1
True 4291 344
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F10
True 6259 596
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F11
True 6102 509
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F12
True 5711 592
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F13
True 6319 711
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F14
True 6359 924
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F15
True 6013 778
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F16
True 6047 885
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F17
True 6109 714
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F18
True 5940 637
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F19
True 6327 676
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F2
True 4588 501
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F20
True 6049 573
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F21
True 5704 576
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F22
True 5618 669
Eve_230929_S3688_CP_Brain_glyco_steppedHCD_F23
True 5596 544
Eve_230929_S3688_CP_Brain_